# The sales dataset

Sales.csv can be found in the files folder. The file contains 100 sales records. Each record contains 6 fields:
* Quantity
* InvoiceDate
* UnitPrice
* CustomerID
* CountryCode

**Tasks**  
Analyze the dataset using only NumPy. 

**Step 1**   
Load the dataset using np.loadtxt() or np.genfromtxt().

In [1]:
import numpy as np

# Load the dataset using genfromtxt (handles mixed data types better)
# Assuming the file has a header row that we want to skip
data = np.genfromtxt('files/sales.csv', delimiter=',', dtype=None, encoding='utf-8', names=True, skip_header=1)

# Display basic information about the loaded data
print("Dataset loaded successfully!")
print(f"Number of records: {len(data)}")
print(f"Field names: {data.dtype.names}")
print("\nFirst 5 records:")
for i in range(min(5, len(data))):
    print(data[i])

Dataset loaded successfully!
Number of records: 99
Field names: ('70', '16909344000', '90', '330930', '50')

First 5 records:
(15.0, 1675468800.0, 37.6, 76105.0, 6.0)
(11.0, 1692057600.0, 24.97, 61885.0, 9.0)
(8.0, 1681171200.0, 80.51, 46631.0, 5.0)
(7.0, 1683763200.0, 47.56, 82991.0, 1.0)
(19.0, 1694649600.0, 98.36, 14014.0, 4.0)


**Step 2**   
Calculate basic statistics for the field UnitPrice:   
These basic statistics include min, max, mean, median and standard deviation.

In [3]:
import numpy as np

# Load the dataset
data = np.genfromtxt('files/sales.csv', delimiter=',', dtype=None, encoding='utf-8', names=True, skip_header=1)

# Print column names to see what's available
print("Column names in dataset:")
print(data.dtype.names)
print()

# Try different possible column name variations
possible_names = ['UnitPrice', 'Unit Price', 'unit_price', 'Price', 'price']

unit_prices = None
for name in possible_names:
    if name in data.dtype.names:
        unit_prices = data[name]
        print(f"Found column: '{name}'")
        break

if unit_prices is None:
    # If still not found, try accessing by index
    print("\nTrying by index...")
    # Assuming UnitPrice might be the 3rd column (index 2)
    # First, let's see the first row to understand structure
    print("First row of data:")
    print(data[0])
    
    # If data is a structured array, we might need to convert to regular array
    data_array = np.array([list(row) for row in data])
    print(f"\nData array shape: {data_array.shape}")
    print("First row as array:")
    print(data_array[0])
    
    # Try to identify which column might be UnitPrice (likely numeric with reasonable values)
    for col_idx in range(data_array.shape[1]):
        try:
            col_values = data_array[:, col_idx].astype(float)
            if np.mean(col_values) > 0 and np.mean(col_values) < 1000:  # UnitPrice likely in this range
                print(f"\nColumn {col_idx} looks like UnitPrice (mean: {np.mean(col_values):.2f})")
                unit_prices = col_values
                break
        except:
            continue

Column names in dataset:
('70', '16909344000', '90', '330930', '50')


Trying by index...
First row of data:
(15.0, 1675468800.0, 37.6, 76105.0, 6.0)

Data array shape: (99, 5)
First row as array:
[1.5000000e+01 1.6754688e+09 3.7600000e+01 7.6105000e+04 6.0000000e+00]

Column 0 looks like UnitPrice (mean: 9.67)


**Step 3**   
Revenue analysis:
* Calculate the overall total revenue
* Calculate the total revenue per country

In [4]:
import numpy as np

# Load the dataset - assuming all columns are numeric
data = np.loadtxt('files/sales.csv', delimiter=',', skiprows=1)

# Assuming columns: Quantity (0), InvoiceDate (1), UnitPrice (2), CustomerID (3), CountryCode (4)
quantities = data[:, 0]
unit_prices = data[:, 2]
country_codes = data[:, 4]

# Calculate revenue per row (Quantity * UnitPrice)
revenue = quantities * unit_prices

# Overall total revenue
total_revenue = np.sum(revenue)

# Revenue per country
unique_countries = np.unique(country_codes)
revenue_per_country = {}

for country in unique_countries:
    # Find rows for this country
    country_mask = (country_codes == country)
    country_revenue = np.sum(revenue[country_mask])
    revenue_per_country[int(country)] = country_revenue

# Display results
print("REVENUE ANALYSIS")
print("=" * 50)
print(f"Overall total revenue: {total_revenue:.2f}")
print()
print("Revenue per country:")
print("-" * 30)
for country, rev in sorted(revenue_per_country.items()):
    print(f"Country {country}: {rev:.2f}")

REVENUE ANALYSIS
Overall total revenue: 55632.42

Revenue per country:
------------------------------
Country 1: 7474.12
Country 2: 5793.04
Country 3: 7779.33
Country 4: 9201.47
Country 5: 9827.68
Country 6: 870.28
Country 7: 6226.82
Country 8: 4070.85
Country 9: 4388.83


**Step 4**   
Customer insights:
* Identify the customer with the highest total spend.
* Identify the customer with the most purchases.

In [5]:
import numpy as np

# Load the dataset
data = np.loadtxt('files/sales.csv', delimiter=',', skiprows=1)

# Assuming columns: Quantity (0), InvoiceDate (1), UnitPrice (2), CustomerID (3), CountryCode (4)
quantities = data[:, 0]
unit_prices = data[:, 2]
customer_ids = data[:, 3].astype(int)  # Convert to int for cleaner display

# Calculate revenue per row
revenue = quantities * unit_prices

# Find unique customers
unique_customers = np.unique(customer_ids)

# Initialize arrays for results
customer_total_spend = []
customer_total_purchases = []

# Calculate for each customer
for customer in unique_customers:
    # Mask for this customer
    customer_mask = (customer_ids == customer)
    
    # Total spend (sum of revenue)
    total_spend = np.sum(revenue[customer_mask])
    customer_total_spend.append((customer, total_spend))
    
    # Total purchases (sum of quantities)
    total_purchases = np.sum(quantities[customer_mask])
    customer_total_purchases.append((customer, total_purchases))

# Convert to numpy arrays for easier manipulation
customer_total_spend = np.array(customer_total_spend)
customer_total_purchases = np.array(customer_total_purchases)

# Find customer with highest spend
highest_spend_idx = np.argmax(customer_total_spend[:, 1])
highest_spend_customer = int(customer_total_spend[highest_spend_idx, 0])
highest_spend_amount = customer_total_spend[highest_spend_idx, 1]

# Find customer with most purchases
most_purchases_idx = np.argmax(customer_total_purchases[:, 1])
most_purchases_customer = int(customer_total_purchases[most_purchases_idx, 0])
most_purchases_quantity = customer_total_purchases[most_purchases_idx, 1]

# Display results
print("=" * 60)
print("CUSTOMER INSIGHTS")
print("=" * 60)
print(f"Customer with highest total spend:")
print(f"  Customer ID: {highest_spend_customer}")
print(f"  Total spend: {highest_spend_amount:.2f}")
print()
print(f"Customer with most purchases:")
print(f"  Customer ID: {most_purchases_customer}")
print(f"  Total items purchased: {int(most_purchases_quantity)}")

CUSTOMER INSIGHTS
Customer with highest total spend:
  Customer ID: 42097
  Total spend: 5162.67

Customer with most purchases:
  Customer ID: 42097
  Total items purchased: 74


**Step 5**   
Time analysis:
* Convert timestamps to NumPy datetime64.
* Group by month and calculate monthly revenue.

In [6]:
import numpy as np

# Load the dataset
data = np.loadtxt('files/sales.csv', delimiter=',', skiprows=1)

# Extract columns
quantities = data[:, 0]
invoice_dates = data[:, 1]  # Assuming timestamp is in column 1
unit_prices = data[:, 2]

# Calculate revenue per row
revenue = quantities * unit_prices

# Convert timestamps to datetime64
# Assuming timestamps are in Unix time (seconds since 1970-01-01)
dates = invoice_dates.astype('datetime64[s]')

# Extract year-month for grouping
year_months = dates.astype('datetime64[M]')

# Get unique year-months
unique_months = np.unique(year_months)

# Calculate monthly revenue
monthly_revenue = []
for month in unique_months:
    month_mask = (year_months == month)
    month_total = np.sum(revenue[month_mask])
    monthly_revenue.append((month, month_total))

# Display results
print("=" * 60)
print("MONTHLY REVENUE ANALYSIS")
print("=" * 60)
print(f"{'Month':<15} {'Revenue':>20}")
print("-" * 60)

for month, rev in monthly_revenue:
    # Convert month to string for display
    month_str = np.datetime_as_string(month)
    print(f"{month_str:<15} {rev:>20,.2f}")

print("=" * 60)

# Additional statistics
all_monthly = np.array([rev for _, rev in monthly_revenue])
print(f"\nMonthly average: {np.mean(all_monthly):.2f}")
print(f"Best month: {monthly_revenue[np.argmax(all_monthly)][0]} with {np.max(all_monthly):.2f}")
print(f"Worst month: {monthly_revenue[np.argmin(all_monthly)][0]} with {np.min(all_monthly):.2f}")

MONTHLY REVENUE ANALYSIS
Month                        Revenue
------------------------------------------------------------
2023-01                     3,140.31
2023-02                     3,677.05
2023-03                     4,349.68
2023-04                     7,037.70
2023-05                     3,961.28
2023-06                     5,051.71
2023-07                     2,182.44
2023-08                    12,732.68
2023-09                     3,524.12
2023-10                     4,619.10
2023-11                     3,255.68
2023-12                     2,100.67

Monthly average: 4636.04
Best month: 2023-08 with 12732.68
Worst month: 2023-12 with 2100.67


**Step 6**   
Based on your initial exploration of the dataset you are now asked to take the next step in your analysis.   
🎯 Your task: Write down three testable hypotheses that could be explored further. Each hypothesis should:

* Be based on patterns, anomalies, or trends you observed in your summary statistics.
* Be specific and measurable.
* Suggest a direction or relationship (e.g., “X is greater than Y”, “X is positively correlated with Y”).

📌 Example hypotheses:

* “Students who study more than 10 hours a week have a higher average test score than those who study less.”
* “There is a significant difference in average income between urban and rural regions.”
* “The standard deviation of product prices is higher in Category A than in Category B.”

In [7]:
import numpy as np

# Load data
data = np.loadtxt('files/sales.csv', delimiter=',', skiprows=1)

# Extract columns
quantities = data[:, 0]
unit_prices = data[:, 2]
customer_ids = data[:, 3].astype(int)
country_codes = data[:, 4].astype(int)
revenue = quantities * unit_prices

# Display basic stats to support hypotheses
print("=" * 70)
print("SALES DATASET - SUMMARY STATISTICS")
print("=" * 70)
print(f"Total records: {len(data)}")
print(f"Unique customers: {len(np.unique(customer_ids))}")
print(f"Unique countries: {len(np.unique(country_codes))}")
print(f"\nQuantity range: {np.min(quantities)} - {np.max(quantities)}")
print(f"UnitPrice range: {np.min(unit_prices):.2f} - {np.max(unit_prices):.2f}")
print(f"Revenue range: {np.min(revenue):.2f} - {np.max(revenue):.2f}")
print(f"Average revenue per transaction: {np.mean(revenue):.2f}")

print("\n" + "=" * 70)
print("THREE TESTABLE HYPOTHESES")
print("=" * 70)

# Hypothesis 1
print("\n📌 HYPOTHESIS 1:")
print("Customers from certain countries have significantly different")
print("average transaction values compared to others.")
print("\n📊 Supporting observation:")
country_means = []
for country in np.unique(country_codes)[:3]:  # Show first 3 as example
    country_rev = revenue[country_codes == country]
    country_means.append(np.mean(country_rev))
    print(f"  Country {country}: mean transaction = {np.mean(country_rev):.2f}")
print(f"\n  Range of country means: {np.min(country_means):.2f} - {np.max(country_means):.2f}")

# Hypothesis 2
print("\n📌 HYPOTHESIS 2:")
print("Frequent buyers have higher average order values.")
print("\n📊 Supporting observation:")
# Calculate transactions per customer
unique_customers = np.unique(customer_ids)
freq_orders = []
infreq_orders = []
for cust in unique_customers:
    cust_rev = revenue[customer_ids == cust]
    avg_order = np.mean(cust_rev)
    if len(cust_rev) >= 3:  # Frequent: 3+ transactions
        freq_orders.append(avg_order)
    else:  # Infrequent
        infreq_orders.append(avg_order)
print(f"  Frequent buyers (3+ transactions): avg order = {np.mean(freq_orders):.2f}")
print(f"  Infrequent buyers: avg order = {np.mean(infreq_orders):.2f}")

# Hypothesis 3
print("\n📌 HYPOTHESIS 3:")
print("Higher prices lead to lower quantities purchased.")
print("\n📊 Supporting observation:")
# Create price brackets
low_price = unit_prices < 10
medium_price = (unit_prices >= 10) & (unit_prices < 50)
high_price = unit_prices >= 50
print(f"  Low price (<£10): avg quantity = {np.mean(quantities[low_price]):.2f}")
print(f"  Medium price (£10-50): avg quantity = {np.mean(quantities[medium_price]):.2f}")
print(f"  High price (>£50): avg quantity = {np.mean(quantities[high_price]):.2f}")

SALES DATASET - SUMMARY STATISTICS
Total records: 100
Unique customers: 73
Unique countries: 9

Quantity range: 1.0 - 19.0
UnitPrice range: 2.79 - 99.63
Revenue range: 15.96 - 1868.84
Average revenue per transaction: 556.32

THREE TESTABLE HYPOTHESES

📌 HYPOTHESIS 1:
Customers from certain countries have significantly different
average transaction values compared to others.

📊 Supporting observation:
  Country 1: mean transaction = 533.87
  Country 2: mean transaction = 724.13
  Country 3: mean transaction = 555.67

  Range of country means: 533.87 - 724.13

📌 HYPOTHESIS 2:
Frequent buyers have higher average order values.

📊 Supporting observation:
  Frequent buyers (3+ transactions): avg order = 506.07
  Infrequent buyers: avg order = 576.72

📌 HYPOTHESIS 3:
Higher prices lead to lower quantities purchased.

📊 Supporting observation:
  Low price (<£10): avg quantity = 6.50
  Medium price (£10-50): avg quantity = 8.80
  High price (>£50): avg quantity = 10.49
